In [ ]:
from bs4 import BeautifulSoup
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
from pathlib import Path
from urllib.parse import urljoin, urlparse
import csv
import certifi
import re
import ssl
import time

base_url = "https://edition.cnn.com/politics"
ssl_context = ssl.create_default_context(cafile=certifi.where())
headers = {"User-Agent": "Mozilla/5.0"}

def fetch_soup(url):
    request = Request(url, headers=headers)
    with urlopen(request, context=ssl_context, timeout=30) as response:
        return BeautifulSoup(response.read(), "html.parser")

landing_page = fetch_soup(base_url)
story_urls = set()
date_path_pattern = re.compile(r"/\d{4}/\d{2}/\d{2}/")

for link in landing_page.select("a[href]"):
    story_url = urljoin(base_url, link["href"])
    parsed_url = urlparse(story_url)
    if (parsed_url.netloc == "edition.cnn.com"
            and date_path_pattern.search(parsed_url.path)
            and parsed_url.query == ""):
        story_urls.add(story_url.split("#")[0])

stories = []
for index, story_url in enumerate(sorted(story_urls), start=1):
    try:
        story_page = fetch_soup(story_url)
        article = story_page.find("article") or story_page.find("main")
        if article is None:
            continue

        title_tag = article.find("h1") or story_page.find("h1")
        title = title_tag.get_text(" ", strip=True) if title_tag else ""
        paragraphs = [
            paragraph.get_text(" ", strip=True)
            for paragraph in article.find_all("p")
        ]
        paragraphs = [text for text in paragraphs if text]
        story_text = " ".join(dict.fromkeys(paragraphs))

        if title and story_text:
            stories.append({
                "source_url": story_url,
                "title": title,
                "text": story_text,
            })
        print(f"Scraped {index}/{len(story_urls)}: {title[:80]}")
    except (HTTPError, URLError, TimeoutError) as error:
        print(f"Skipped {story_url}: {error}")
    time.sleep(0.5)

output_path = Path("../../data/raw/cnn_politics_stories.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=["source_url", "title", "text"])
    writer.writeheader()
    writer.writerows(stories)

print(f"Discovered {len(story_urls)} story links")
print(f"Saved {len(stories)} stories to {output_path}")

In [7]:
import pandas as pd

df = pd.read_csv("../../data/raw/cnn_politics_stories.csv")
df.head(50)

,source_url,title,text
0,https://edition.cnn.com/2021/01/08/politics/ga...,In pictures: President Donald Trump,"Donald Trump, the 45th president of the United..."
1,https://edition.cnn.com/2025/11/02/politics/ti...,A timeline of US strikes on boats that have ki...,The US military has killed at least 227 people...
2,https://edition.cnn.com/2025/12/11/politics/kf...,‘Rehashed crap’: Former Obama White House atto...,When billionaire sex offender Jeffrey Epstein ...
3,https://edition.cnn.com/2025/12/23/politics/wh...,What voters say they want for Christmas,One big thing dominates Tonya Rincon’s holiday...
4,https://edition.cnn.com/2026/01/09/politics/20...,CNN’s guide to the most important elections of...,This year’s midterms will not only shape the f...
5,https://edition.cnn.com/2026/01/19/politics/io...,"A year into Trump’s second term, Iowa voters o...","Shanen Ebersole is smiling, keeping an eye on ..."
6,https://edition.cnn.com/2026/02/17/politics/ga...,Jesse Jackson’s life in pictures,"The Rev. Jesse Jackson, the civil rights icon ..."
7,https://edition.cnn.com/2026/02/21/politics/ec...,Analysis: These two new economic numbers blew ...,"Two new pieces of economic data, one released ..."
8,https://edition.cnn.com/2026/02/24/politics/fa...,Fact check: Trump makes false claims about the...,President Donald Trump made numerous false or ...
9,https://edition.cnn.com/2026/04/01/politics/fe...,‘Haters gonna hate’: A top FEMA official defen...,The top official in charge of the federal gove...


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   source_url  50 non-null     str  
 1   title       50 non-null     str  
 2   text        50 non-null     str  
dtypes: str(3)
memory usage: 1.3 KB


In [12]:
cleaned_df = df.drop(columns=["source_url", "title"]).dropna().reset_index(drop=True)
cleaned_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    50 non-null     str  
dtypes: str(1)
memory usage: 532.0 bytes


In [14]:
cleaned_df.to_csv("../../data/processed/cnn_politics_stories_cleaned.csv", index=False)